In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
from scipy.fft import fftn, ifftn, fftfreq
import illustris_python as il
from matplotlib.colors import LogNorm
import matplotlib.animation as animation
import matplotlib.patches as mpatches
import os
import matplotlib.ticker as ticker

In [ ]:
z_list = [0.0,0.1,0.2,0.3,0.4,0.5,0.7,1.0,1.5,2.0,3.01,4.01,5.0,6.01,7.01,8.01,9.0,10.0]
z_smol_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0, 1.5, 2.0, 3.01, 4.01, 5.0, 6.01, 8.01, 10.0]
z_list_cute = [0.0,2.0,5.0,10.0]
snap_list = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,11,8,6,4]
snap_smol_list = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,8,4]
smol_id = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,15,17]

colors = plt.colormaps['viridis'](np.linspace(0, 1, 4))
colors[3] = [1.0, 0.65, 0.0, 1.0]

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    'font.size': 12,
})

basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (f"The voxel width is {dl} cMpc/h")

norm_dens = []
voids = []
filaments = []
nodes = []
walls = []
dens = []
v_flds = []
v_flds_sum = []
strm_flds = []


with h5py.File('/Users/users/roana/roana/BSc_Thesis/nexus_outputs.h5', 'r') as f:
    for red in z_list:
        norm_dens.append(np.transpose(f[f'z_{red}']['normalized_density'][:], (1,2,0)))
        dens.append(np.transpose(f[f'z_{red}']['density'][:], (1,2,0)))
        voids.append(np.transpose(f[f'z_{red}']['voids'][:], (1,2,0)))
        filaments.append(np.transpose(f[f'z_{red}']['filaments'][:], (1,2,0)))
        nodes.append(np.transpose(f[f'z_{red}']['nodes'][:], (1,2,0)))
        walls.append(np.transpose(f[f'z_{red}']['walls'][:], (1,2,0)))
        print(f"Suessfully Loaded Stuff for z={red}")

n = len(norm_dens)
print(f"{n} snapshots")

with h5py.File('/Users/users/roana/roana/BSc_Thesis/nexus_outputs.h5', 'r') as f:
    for red in z_list_cute:
        
        # 1. Load the raw arrays into memory first
        v_raw = f[f'z_{red}']['vel_fld'][:]
        v_sum_raw = f[f'z_{red}']['vel_fld_sum'][:]
        strm_raw = f[f'z_{red}']['stream_fld'][:]
        
        # --- Handle Velocity Field (4D) ---
        if v_raw.ndim == 4:
            # If Julia saved it as (3, X, Y, Z), h5py reads it as (Z, Y, X, 3) or (3, Z, Y, X)
            # We transpose the spatial axes but leave the vector axis in place.
            if v_raw.shape[0] == 3: # Shape is (3, 128, 128, 128)
                v_flds.append(np.transpose(v_raw, (0, 2, 3, 1))) 
            else:                   # Shape is (128, 128, 128, 3)
                v_flds.append(np.transpose(v_raw, (1, 2, 0, 3)))
        else:
            v_flds.append(np.transpose(v_raw, (1, 2, 0)))

        # --- Handle Velocity Sum Field ---
        if v_sum_raw.ndim == 4:
            if v_sum_raw.shape[0] == 3:
                v_flds_sum.append(np.transpose(v_sum_raw, (0, 2, 3, 1)))
            else:
                v_flds_sum.append(np.transpose(v_sum_raw, (1, 2, 0, 3)))
        else:
            v_flds_sum.append(np.transpose(v_sum_raw, (1, 2, 0)))

        # --- Handle Stream Field (Likely 3D, but safe to check) ---
        if strm_raw.ndim == 4:
            if strm_raw.shape[0] == 3:
                strm_flds.append(np.transpose(strm_raw, (0, 2, 3, 1)))
            else:
                strm_flds.append(np.transpose(strm_raw, (1, 2, 0, 3)))
        else:
            strm_flds.append(np.transpose(strm_raw, (1, 2, 0)))

        print(f"Successfully Loaded Kinematics for z={red}")


In [ ]:
struct = ["Everything","Nodes","Filaments","Walls","Voids"]

cos_all = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/cos_all.npy")
cos_avg = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/cos_avg.npy")
cos_err_up = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/cos_err_up.npy")
cos_err_down = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/cos_err_down.npy")

mag_all = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/mag_all.npy")
mag_avg = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/mag_avg.npy")
mag_err_up = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/mag_err_up.npy")
mag_err_down = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/mag_err_down.npy")

diff_all = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/diff_all.npy")
diff_avg = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/diff_avg.npy")
diff_err_up = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/diff_err_up.npy")
diff_err_down = np.load("/Users/users/roana/roana/BSc_Thesis/Useful_Python/diff_err_down.npy")

In [ ]:
red = [ 20.05, 14.99, 11.98, 10.98, 10.00, 9.39, 9.00, 8.45, 8.01, 7.60, 7.24, 7.01, 6.49, 6.01,
    5.85, 5.53, 5.23, 5.00, 4.66, 4.43, 4.18, 4.01, 3.71, 3.49,
    3.28, 3.01, 2.90, 2.73, 2.58, 2.44, 2.32, 2.21, 2.10, 2.00,
    1.90, 1.82, 1.74, 1.67, 1.60, 1.53, 1.50, 1.41, 1.36, 1.30,
    1.25, 1.21, 1.15, 1.11, 1.07, 1.04, 1.00, 0.95, 0.92, 0.89,
    0.85, 0.82, 0.79, 0.76, 0.73, 0.70, 0.68, 0.64, 0.62, 0.60,
    0.58, 0.55, 0.52, 0.50, 0.48, 0.46, 0.44, 0.42, 0.40, 0.38,
    0.36, 0.35, 0.33, 0.31, 0.30, 0.27, 0.26, 0.24, 0.23, 0.21,
    0.20, 0.18, 0.17, 0.15, 0.14, 0.13, 0.11, 0.10, 0.08, 0.07,
    0.06, 0.05, 0.03, 0.02, 0.01, 0.00
]

In [ ]:
tng_ages = [13.624, 13.532,
    13.433, 13.385, 13.328, 13.286, 13.256, 13.207, 13.163, 13.118, 13.071, 13.039, 
    12.960, 12.871, 12.839, 12.767, 12.691, 12.628, 12.521, 12.437, 12.337, 12.260, 
    12.115, 11.991, 11.859, 11.666, 11.585, 11.419, 11.264, 11.116, 10.961, 10.826, 
    10.674, 10.519, 10.356, 10.210, 10.059, 9.901, 9.766, 9.597, 9.470, 9.301, 
    9.147, 8.987, 8.823, 8.688, 8.514, 8.372, 8.226, 8.077, 7.925, 7.730, 
    7.610, 7.447, 7.281, 7.111, 6.981, 6.808, 6.671, 6.489, 6.345, 6.161, 
    6.017, 5.872, 5.724, 5.533, 5.371, 5.216, 5.060, 4.901, 4.741, 4.578, 
    4.414, 4.247, 4.079, 3.966, 3.764, 3.621, 3.504, 3.269, 3.149, 2.969, 
    2.787, 2.665, 2.480, 2.294, 2.169, 1.979, 1.852, 1.660, 1.466, 1.336, 
    1.140, 1.008, 0.810, 0.678, 0.475, 0.343, 0.136, 0.000
]

age = 13.803

for k in range(100):
    tng_ages[k] = age - tng_ages[k]


In [ ]:
fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(121)


styles = ["solid","-.","--",":"]

ax.plot(red, cos_avg[0], label = f"{struct[0]}", c="black")
ax.fill_between(red, (np.array(cos_avg[0])-np.array(cos_err_down[0])), 
                (np.array(cos_avg[0])+np.array(cos_err_up[0])), alpha=.2, color="black")
    
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"cos$\theta$")
ax.legend()
ax.grid()

ax = fig.add_subplot(122)

for k in range(1,5):
    ax.plot(red[9:], cos_avg[k][9:], label = f"{struct[k]}", linestyle = styles[k-1], c = colors[k-1])

        
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"cos$\theta$")
ax.legend()
ax.grid()

plt.tight_layout()

plt.savefig("cos.png", transparent = True, bbox_inches="tight")
plt.savefig("cos.pdf", transparent = True, bbox_inches="tight");

In [ ]:
fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(121)


styles = ["solid","-.","--",":"]

ax.plot(tng_ages, cos_avg[0], label = f"{struct[0]}", c="black")
ax.fill_between(tng_ages, (np.array(cos_avg[0])-np.array(cos_err_down[0])), 
                (np.array(cos_avg[0])+np.array(cos_err_up[0])), alpha=.2, color="black")
    
ax.set_xlabel("Age [Gyr]")
ax.set_ylabel(r"cos$\theta$")
ax.legend()
ax.grid()

ax = fig.add_subplot(122)

for k in range(1,5):
    ax.plot(tng_ages[9:], cos_avg[k][9:], label = f"{struct[k]}", linestyle = styles[k-1], c = colors[k-1])

        
ax.set_xlabel("Age [Gyr]")
ax.set_ylabel(r"cos$\theta$")
ax.legend()
ax.grid()

plt.tight_layout()

plt.savefig("cos.png", transparent = True, bbox_inches="tight")
plt.savefig("cos.pdf", transparent = True, bbox_inches="tight");

In [ ]:
cos_bins = np.linspace(-1, 1, 100) 
heatmap_data = np.zeros((len(cos_bins) - 1, 100))

for t in range(100):

    hist, _ = np.histogram(cos_all[0][t], bins=cos_bins)
    heatmap_data[:, t] = hist

fig, ax = plt.subplots(figsize=(8, 5))

red_arr=np.array(red)

red_centers = (red_arr[:-1] + red_arr[1:]) / 2
cos_centers = (cos_bins[:-1] + cos_bins[1:]) / 2

# Trim your data matrix to 99 columns if it's currently 100
# (or ensure it's generated as shape (9, 99))
pcm = ax.pcolormesh(
    red_centers, 
    cos_centers, 
    heatmap_data[:, :-1] + 1,  # Trims off the extra 100th column to match the 99 centers
    shading='auto', 
    cmap='viridis', norm = LogNorm()
)

ax.set_ylabel(r"$\cos \theta$")
ax.set_xlabel(r"Redshift $z$")

ax.set_xlim(red_arr.max(), red_arr.min())
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(f"Particle Count");

In [ ]:
cos_bins = np.linspace(-1, 1, 100) 
heatmap_data = np.zeros((len(cos_bins) - 1, 100))

for t in range(100):

    hist, _ = np.histogram(cos_all[0][t], bins=cos_bins)
    heatmap_data[:, t] = hist

fig, ax = plt.subplots(figsize=(8, 5))

age_arr=np.array(tng_ages)

age_centers = (age_arr[:-1] + age_arr[1:]) / 2
cos_centers = (cos_bins[:-1] + cos_bins[1:]) / 2

# Trim your data matrix to 99 columns if it's currently 100
# (or ensure it's generated as shape (9, 99))
pcm = ax.pcolormesh(
    age_centers, 
    cos_centers, 
    heatmap_data[:, :-1] + 1,  # Trims off the extra 100th column to match the 99 centers
    shading='auto', 
    cmap='viridis', norm = LogNorm()
)

ax.set_ylabel(r"$\cos \theta$")
ax.set_xlabel("Age [Gyr]")

ax.set_xlim(age_arr.min(), age_arr.max())
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(f"Particle Count");

In [ ]:
dens_bins = np.linspace(np.min(np.log10(np.array(norm_dens).ravel())), np.max(np.log10(np.array(norm_dens).ravel())), 100) 
heatmap_data = np.zeros((len(dens_bins) - 1, 18))

for t in range(18):

    hist, _ = np.histogram(np.log10(norm_dens[t].ravel()), bins=dens_bins)
    heatmap_data[:, t] = hist

fig, ax = plt.subplots(figsize=(8, 5))

red_arr=np.array(z_list)

red_centers = (red_arr[:-1] + red_arr[1:]) / 2
dens_centers = (dens_bins[:-1] + dens_bins[1:]) / 2

# Trim your data matrix to 99 columns if it's currently 100
# (or ensure it's generated as shape (9, 99))
pcm = ax.pcolormesh(
    red_centers, 
    dens_centers, 
    heatmap_data[:, :-1] + 1,  # Trims off the extra 100th column to match the 99 centers
    shading='auto', 
    cmap='magma', norm = LogNorm()
)

ax.plot(z_list, np.log10(np.mean(np.array(norm_dens), axis = (1,2,3))), color = "black", linestyle = ":", label = "Average")

ax.set_ylabel(r"$\log (1+\delta)$")
ax.set_xlabel(r"Redshift $z$")
ax.legend()

ax.set_xlim(red_arr.min(), red_arr.max())
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(f"Pixel Count");

In [ ]:
np.shape(np.mean(np.array(norm_dens), axis = (1,2,3)))

In [ ]:
print(np.mean(np.array(norm_dens), axis = (1,2,3)))

In [ ]:
mag_bins = np.linspace(np.min(mag_all[0]), np.max(mag_all[0]), 100) 
heatmap_data = np.zeros((len(mag_bins) - 1, 100))

for t in range(100):

    hist, _ = np.histogram(mag_all[0][t], bins=mag_bins)
    heatmap_data[:, t] = hist

fig, ax = plt.subplots(figsize=(8, 5))


# Trim your data matrix to 99 columns if it's currently 100
# (or ensure it's generated as shape (9, 99))
pcm = ax.pcolormesh(
    age_centers, 
    mag_centers, 
    heatmap_data[:, :-1] + 1,  # Trims off the extra 100th column to match the 99 centers
    shading='auto', 
    cmap='viridis', norm = LogNorm()
)

ax.set_ylabel(r"$|v_x|/|v_q|$")
ax.set_xlabel("Age [Gyr]")

ax.set_xlim(age_arr.min(), age_arr.max())
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(f"Particle Count");

In [ ]:
fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(121)


styles = ["solid","-.","--",":"]

ax.plot(red, mag_avg[0], label = f"{struct[0]}", c="black")
ax.fill_between(red, (np.array(mag_avg[0])-np.array(mag_err_down[0])), 
                (np.array(mag_avg[0])+np.array(mag_err_up[0])), alpha=.2, color="black")
    
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"$|v_x|/|v_q|$")
ax.legend()
ax.grid()

ax = fig.add_subplot(122)

for k in range(1,5):
    ax.plot(red, mag_avg[k], label = f"{struct[k]}", linestyle = styles[k-1], c = colors[k-1])

        
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"$|v_x|/|v_q|$")
ax.legend()
ax.grid()

plt.tight_layout()

plt.savefig("mag.png", transparent = True, bbox_inches="tight")
plt.savefig("mag.pdf", transparent = True, bbox_inches="tight");

In [ ]:
fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(121)


styles = ["solid","-.","--",":"]

ax.plot(tng_ages, mag_avg[0], label = f"{struct[0]}", c="black")
ax.fill_between(tng_ages, (np.array(mag_avg[0])-np.array(mag_err_down[0])), 
                (np.array(mag_avg[0])+np.array(mag_err_up[0])), alpha=.2, color="black")
    
ax.set_xlabel("Age [Gyr]")
ax.set_ylabel(r"$|v_x|/|v_q|$")
ax.legend()
ax.grid()

ax = fig.add_subplot(122)

for k in range(1,5):
    ax.plot(tng_ages[9:], mag_avg[k][9:], label = f"{struct[k]}", linestyle = styles[k-1], c = colors[k-1])

        
ax.set_xlabel("Age [Gyr]")
ax.set_ylabel(r"$|v_x|/|v_q|$")
ax.legend()
ax.grid()

plt.tight_layout()

plt.savefig("mag.png", transparent = True, bbox_inches="tight")
plt.savefig("mag.pdf", transparent = True, bbox_inches="tight");

In [ ]:
plt.imshow(np.log10(strm_flds[0][:,:,67]), origin="lower");

In [ ]:
zzz = 55

fig = plt.figure()

global_max = np.max([np.log10(strm_flds[i][:, :, zzz] + 1) for i in range(4)])

for i in range(4):
    ax = fig.add_subplot(2,2,i+1)
    im = ax.imshow(np.log10(strm_flds[i][:, :, zzz] + 1), origin="lower", extent=[0, L, 0, L], 
               cmap='managua_r', vmin=0, vmax=global_max)
    fig.colorbar(im, ax = ax, label = r"$\log$ (\#streams)")
    ax.set_title(fr"$z = {z_list_cute[i]}$")
    ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))

plt.tight_layout();

In [ ]:
z_min = 0.0
z_max = 12.5

# Ensure L and res are defined (e.g., L=35.0, res=128)
grid_z_min = int(np.floor((z_min / L) * res)) % res
grid_z_max = int(np.floor((z_max / L) * res)) % res

fig = plt.figure(figsize = (10,8))

global_max = np.max([np.log10(np.mean(strm_flds[i][:, :, grid_z_min:grid_z_max],axis=2) + 1) for i in range(4)])

for i in range(4):
    ax = fig.add_subplot(2,2,i+1)
    im = ax.imshow(np.log10(np.mean(strm_flds[i][:, :, grid_z_min:grid_z_max],axis=2) + 1), origin="lower", extent=[0, L, 0, L], 
               cmap='cubehelix', vmin=0, vmax=global_max)
    fig.colorbar(im, ax = ax, label = r"$\log$ (\#streams + 1)")
    ax.set_title(fr"$z = {z_list_cute[i]}$")
    ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.set_xlabel("x [cMpc/h]")
    ax.set_ylabel("y [cMpc/h]")

plt.tight_layout()

plt.savefig("multi.png", transparent=True, bbox_inches="tight")
plt.savefig("multi.pdf", transparent=True, bbox_inches="tight");

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z_min = 0.0
z_max = 12.5

# Ensure L and res are defined (e.g., L=35.0, res=128)
grid_z_min = int(np.floor((z_min / L) * res)) % res
grid_z_max = int(np.floor((z_max / L) * res)) % res

# 1. Use layout="constrained" to perfectly fit the global colorbar
fig, axes = plt.subplots(2, 2, figsize=(10, 8), layout="constrained")
axes = axes.flatten()

# Keep your original log10 global max calculation
global_max = np.max([np.log10(np.mean(strm_flds[i][:, :, grid_z_min:grid_z_max], axis=2) + 1) for i in range(4)])

for i in range(4):
    ax = axes[i]
    
    # Keep your original log10 plotting data
    data = np.log10(np.mean(strm_flds[i][:, :, grid_z_min:grid_z_max], axis=2) + 1)
    
    im = ax.imshow(data, origin="lower", extent=[0, L, 0, L], 
                   cmap='cubehelix', vmin=0, vmax=global_max)
    
    # Put the title inside the box so we don't waste space
    ax.text(0.04, 0.96,                  
            f"$z = {z_list_cute[i]}$",   
            transform=ax.transAxes,      
            fontsize=20, 
            color="black", 
            verticalalignment="top", 
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=5))
            
    # Completely remove all axes/ticks as requested
    ax.set_xticks([])
    ax.set_yticks([])

# 2. Add ONE global colorbar on the right
cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.9, aspect=30, pad=0.02)
cbar.set_label(r"$\log (\#\mathrm{streams} + 1)$", fontsize=20)
cbar.ax.tick_params(labelsize=15)

# (plt.tight_layout() removed because constrained layout handles it)

plt.savefig("multi.png", transparent=True, bbox_inches="tight", dpi=300)
plt.savefig("multi.pdf", transparent=True, bbox_inches="tight")

In [ ]:
fig = plt.figure(figsize=(9,8))
ax = fig.add_subplot(111)

slice_z = 13

im = ax.imshow(np.log10(norm_dens[0][:,:,slice_z]), origin = "lower",cmap = 'RdYlBu_r',extent=[0, L, 0, L])

step = 4

X = np.linspace(0, 35, 128)[::step]
Y = np.linspace(0, 35, 128)[::step]

U_raw = v_flds[0][0][::step, ::step, slice_z]
V_raw = v_flds[0][1][::step, ::step, slice_z]

# THE SANITIZER: Kill the NaNs and masked values before doing any math!
import numpy.ma as ma
U = np.nan_to_num(ma.filled(U_raw, 0.0))
V = np.nan_to_num(ma.filled(V_raw, 0.0))

# 1. Calculate the Magnitude (Speed)
M = np.hypot(U, V)

# 2. Normalize the vectors
M_safe = np.where(M == 0, 1, M)
U_norm = U / M_safe
V_norm = V / M_safe

log_M = np.log10(M + 1)
U_log = U_norm * log_M
V_log = V_norm * log_M

# 3. Create the Custom RGBA Array for Opacity
# THE FIX: Use np.nanmax instead of np.max, and prevent divide-by-zero
max_log_M = np.nanmax(log_M)
if max_log_M == 0:
    max_log_M = 1.0  # Failsafe for completely empty slices

alpha_norm = log_M / max_log_M
alphas = 0.15 + 0.85 * alpha_norm 

# Create an array of shape (N, 4) for RGBA
rgba_colors = np.zeros((U_log.size, 4))
rgba_colors[:, 0] = 0  # Red
rgba_colors[:, 1] = 0  # Green
rgba_colors[:, 2] = 0  # Blue
rgba_colors[:, 3] = alphas.flatten()  # The Alpha channel!

# 4. Plot the Quiver (Notice we removed the 5th color argument)
q = ax.quiver(X, Y, U_log, V_log, 
              color=rgba_colors,   # Solid color to pop against the background
              scale=75,        # Tweak this: smaller number = longer arrows
              width=0.003,     
              headwidth=4,     
              headlength=5,    
              pivot='mid')

# 5. Bring back the Quiver Key! 
# Since we lost the colorbar for the arrows, we need a reference arrow.
# Let's say we want a reference arrow representing 100 km*sqrt(a)/s:
ref_speed = 100
ax.quiverkey(q, X=0.85, Y=1.03, U=np.log10(ref_speed + 1), 
             label=f'{ref_speed} km $\\sqrt{{a}}$/s', 
             labelpos='E', coordinates='axes', color='black')

# 6. Put the colorbar back to where it belongs (the density field)
# Note: I assumed your imshow is saved to a variable called 'im' 
# e.g., im = ax.imshow(...)
cbar = fig.colorbar(im, ax=ax, label=r"$\log (1+\delta)$")

ax.set_xlabel("x [cMpc/h]")
ax.set_ylabel("y [cMpc/h]")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


In [ ]:
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111)

slice_z = 13

ax.imshow(np.log10(norm_dens[0][:,:,slice_z]), origin = "lower",cmap = 'Greys_r',extent=[0, L, 0, L])

step = 4

X = np.linspace(0, 35, 128)[::step]
Y = np.linspace(0, 35, 128)[::step]

U = v_flds[0][0][::step, ::step, slice_z]
V = v_flds[0][1][::step, ::step, slice_z]

# 1. Calculate the Magnitude (Speed)
M = np.hypot(U, V)  # np.hypot is a faster, safer way to do sqrt(U^2 + V^2)

# 2. Normalize the vectors to a constant length of 1
# We use np.where to prevent "Divide by Zero" errors in empty voids
M_safe = np.where(M == 0, 1, M)
U_norm = U / M_safe
V_norm = V / M_safe

# 3. Pass M as the 5th argument (the Color array)
# Because U_norm and V_norm are all length 1, we can use a fixed scale
q = ax.quiver(X, Y, U_norm, V_norm, np.log10(M), 
              cmap='inferno',    # The colormap for the magnitudes
              scale=40,        # Adjust this to change the constant length of ALL arrows
              width=0.003,     
              headwidth=4,     
              headlength=5,    
              pivot='mid')

# 4. We don't need a quiverkey anymore! We need a colorbar.
cbar = fig.colorbar(q, ax=ax, label=r"$\log |v|$ [km $\sqrt{a}$/s]")

ax.set_xlabel("x [cMpc/h]")
ax.set_ylabel("y [cMpc/h]")
ax.set_aspect("equal")
plt.tight_layout();



In [ ]:
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111)

slice_z = 13

ax.imshow(np.log10(norm_dens[0][:,:,slice_z]), origin = "lower",cmap = 'Greys',extent=[0, L, 0, L])

step = 4

X = np.linspace(0, 35, 128)[::step]
Y = np.linspace(0, 35, 128)[::step]

U = v_flds[0][0][::step, ::step, slice_z]
V = v_flds[0][1][::step, ::step, slice_z]

# 1. Calculate the Magnitude (Speed)
M = np.hypot(U, V)  # np.hypot is a faster, safer way to do sqrt(U^2 + V^2)

# 2. Normalize the vectors to a constant length of 1
# We use np.where to prevent "Divide by Zero" errors in empty voids
M_safe = np.where(M == 0, 1, M)
U_norm = U / M_safe
V_norm = V / M_safe

# 3. Pass M as the 5th argument (the Color array)
# Because U_norm and V_norm are all length 1, we can use a fixed scale
q = ax.quiver(X, Y, U_norm, V_norm, np.log10(M), 
              cmap='viridis',    # The colormap for the magnitudes
              scale=40,        # Adjust this to change the constant length of ALL arrows
              width=0.003,     
              headwidth=4,     
              headlength=5,    
              pivot='mid')

# 4. We don't need a quiverkey anymore! We need a colorbar.
cbar = fig.colorbar(q, ax=ax, label=r"$\log |v|$ [km $\sqrt{a}$/s]")

ax.set_xlabel("x [cMpc/h]")
ax.set_ylabel("y [cMpc/h]")
ax.set_aspect("equal")
plt.tight_layout();



In [ ]:
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111)

slice_z = 13

im = ax.imshow(np.log10(norm_dens[0][:,:,slice_z]), origin = "lower",cmap = 'RdYlBu_r',extent=[0, L, 0, L])

step = 4

X = np.linspace(0, 35, 128)[::step]
Y = np.linspace(0, 35, 128)[::step]

U = v_flds_sum[0][0][::step, ::step, slice_z]
V = v_flds_sum[0][1][::step, ::step, slice_z]

M = np.hypot(U, V)  # np.hypot is a faster, safer way to do sqrt(U^2 + V^2)

ax.streamplot(X, Y, U, V, linewidth = np.log10(M)/(0.5*np.max(np.log10(M))) , color = "black", density=3)

ax.set_aspect('equal')

cbar = fig.colorbar(im, ax=ax, label=r"$\log (1+\delta)$")

ax.set_xlabel("x [cMpc/h]")
ax.set_ylabel("y [cMpc/h]")
ax.set_aspect("equal")
plt.tight_layout();



In [ ]:
fig = plt.figure(figsize=(10,8))

cute_id = [0, 9, 12, 17]

slice_z = 13

step = 4

global_max = np.max([np.log10(norm_dens[cute_id[i]][:,:,slice_z]) for i in range(4)])
global_min = np.min([np.log10(norm_dens[cute_id[i]][:,:,slice_z]) for i in range(4)])

for i in range(4):
    ax = fig.add_subplot(2,2,i+1)
    im = ax.imshow(np.log10(norm_dens[cute_id[i]][:,:,slice_z]), origin = "lower",cmap = 'RdYlBu_r',extent=[0, L, 0, L], vmin = global_min, vmax=global_max)

    X = np.linspace(0, 35, 128)[::step]
    Y = np.linspace(0, 35, 128)[::step]

    U = v_flds_sum[i][0][::step, ::step, slice_z]
    V = v_flds_sum[i][1][::step, ::step, slice_z]

    M = np.hypot(U, V)  # np.hypot is a faster, safer way to do sqrt(U^2 + V^2)

    ax.streamplot(X, Y, U, V, linewidth = np.log10(M)/(0.5*np.max(np.log10(M))) , color = "black", density=1.5)

    ax.set_aspect('equal')

    cbar = fig.colorbar(im, ax=ax, label=r"$\log (1+\delta)$")

    ax.set_xlabel("x [cMpc/h]")
    ax.set_ylabel("y [cMpc/h]")
    ax.set_aspect("equal")

    ax.set_title(fr"$z={z_list_cute[i]}$")

plt.tight_layout();



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 14))
axes = axes.flatten() # Flattens the 2x2 grid into a 1D list so we can loop through it

cute_id = [0, 9, 12, 17]
z_list_cute = [0.0, 2.0, 5.0, 10.0] # Assuming this from your earlier code
slice_z = 13
step = 5

# --- STEP 1: CALCULATE GLOBAL LIMITS ---
# Density limits
global_min = np.min([np.log10(norm_dens[cute_id[i]][:,:,slice_z]) for i in range(4)])
global_max = np.max([np.log10(norm_dens[cute_id[i]][:,:,slice_z]) for i in range(4)])

# Velocity limits (We have to extract M for all 4 snaps first to find the max/min)
v_global_min = np.inf
v_global_max = -np.inf
M_arrays = [] 

for i in range(4):
    U = v_flds_sum[i][0][::step, ::step, slice_z]
    V = v_flds_sum[i][1][::step, ::step, slice_z]
    M = np.hypot(U, V)
    
    # We use M + 1 to prevent taking the log of 0 (empty voids)
    log_M = np.log10(M + 1)
    M_arrays.append((U, V, M, log_M)) # Store them so we don't recalculate
    
    v_global_max = max(v_global_max, np.nanmax(log_M))
    v_global_min = min(v_global_min, np.nanmin(log_M))

# --- STEP 2: PLOT THE GRIDS ---
for i in range(4):
    ax = axes[i]
    
    # Plot Density
    im = ax.imshow(np.log10(norm_dens[cute_id[i]][:,:,slice_z]), origin="lower", 
                   cmap='Greys_r', extent=[0, L, 0, L], 
                   vmin=global_min, vmax=global_max)

    # Retrieve the math we already did
    U, V, M, log_M = M_arrays[i]
    
    X = np.linspace(0, 35, 128)[::step]
    Y = np.linspace(0, 35, 128)[::step]

    M_safe = np.where(M == 0, 1, M)
    U_norm = U / M_safe
    V_norm = V / M_safe

    # Plot Quiver (Applying the global color limits via 'clim')
    q = ax.quiver(X, Y, U_norm, V_norm, log_M, 
                  cmap='viridis',    
                  clim=(v_global_min, v_global_max), # Force global limits here!
                  scale=40,        
                  width=0.003,     
                  headwidth=4,     
                  headlength=5,    
                  pivot='mid')

    ax.set_xlabel("x [cMpc/h]")
    ax.set_ylabel("y [cMpc/h]")
    ax.set_aspect("equal")
    ax.set_title(fr"$z={z_list_cute[i]}$")

# --- STEP 3: THE GLOBAL COLORBARS ---
# Notice we pass 'ax=axes.tolist()' - this tells Matplotlib to scale the colorbar 
# relative to the ENTIRE 2x2 grid, preserving your perfect symmetry!

# Put the Density colorbar on the 
cbar_dens = fig.colorbar(im, ax=axes.tolist(), orientation='horizontal', location='top',
                         shrink=0.8, pad=0.02, label=r"$\log (1+\delta)$")

# Put the Velocity colorbar on the 
cbar_vel = fig.colorbar(q, ax=axes.tolist(), orientation='horizontal', 
                        shrink=0.8, pad=0.08, label=r"$\log |v|$ [km $\sqrt{a}$/s]")

plt.show()

In [ ]:
# 1. ADD layout="constrained" HERE. This perfectly handles global colorbars.
fig, axes = plt.subplots(2, 2, figsize=(14, 12), layout="constrained")
axes = axes.flatten() 

cute_id = [0, 9, 12, 17]
z_list_cute = [0.0, 2.0, 5.0, 10.0] 
slice_z = 13
step = 5
L = 35 # Assuming L is defined somewhere in your code

# --- STEP 1: CALCULATE GLOBAL LIMITS ---
global_min = np.min([np.log10(norm_dens[cute_id[i]][:,:,slice_z]) for i in range(4)])
global_max = np.max([np.log10(norm_dens[cute_id[i]][:,:,slice_z]) for i in range(4)])

v_global_min = np.inf
v_global_max = -np.inf
M_arrays = [] 

for i in range(4):
    U = v_flds_sum[i][0][::step, ::step, slice_z]
    V = v_flds_sum[i][1][::step, ::step, slice_z]
    M = np.hypot(U, V)
    
    log_M = np.log10(M)
    M_arrays.append((U, V, M, log_M, M)) 
    
    v_global_max = max(v_global_max, np.nanmax(log_M))
    v_global_min = min(v_global_min, np.nanmin(log_M))

# --- STEP 2: PLOT THE GRIDS ---
for i in range(4):
    ax = axes[i]
    
    # Plot Density
    im = ax.imshow(np.log10(norm_dens[cute_id[i]][:,:,slice_z]), origin="lower", 
                   cmap='RdYlBu_r', extent=[0, L, 0, L], 
                   vmin=global_min, vmax=global_max, alpha=0.85)

    U, V, M, log_M, M_val = M_arrays[i]
    
    X_grid, Y_grid = np.meshgrid(np.linspace(0, 35, 128)[::step], 
                                 np.linspace(0, 35, 128)[::step])
    
    # Plot Streamlines
    strm = ax.streamplot(X_grid, Y_grid, U, V, 
                         color="black",           
                         norm=plt.Normalize(vmin=v_global_min, vmax=v_global_max),
                         linewidth=M_val/200,          
                         density=1.75)            

    ax.set_aspect("equal")

    ax.text(0.04, 0.96,                  
            f"$z = {z_list_cute[i]}$",   
            transform=ax.transAxes,      
            fontsize=20, 
            color="white", 
            verticalalignment="top", 
            bbox=dict(facecolor="black", alpha=0.75, edgecolor="none", pad=5))

    ax.set_xticks([])
    ax.set_yticks([])

# --- STEP 3: THE RIGHT WAY TO ADD THE GLOBAL COLORBAR ---
# Use fig.colorbar, pass the 1D list of axes. Constrained layout handles the padding natively.
cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.9, aspect=30)
cbar.set_label(r"$\log (1+\delta)$", fontsize=20)
cbar.ax.tick_params(labelsize=15) # Optional: Makes the numbers on the colorbar larger

# REMOVED: plt.tight_layout() -- Do not use this when layout="constrained" is active.

plt.savefig("streams.png", transparent=True, bbox_inches="tight", dpi=300)
plt.savefig("streams.pdf", transparent=True, bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten() # Flattens the 2x2 grid into a 1D list so we can loop through it

cute_id = [0, 9, 12, 17]
z_list_cute = [0.0, 2.0, 5.0, 10.0] # Assuming this from your earlier code
slice_z = 13
step = 5

# --- STEP 1: CALCULATE GLOBAL LIMITS ---
# Density limits

# Velocity limits (We have to extract M for all 4 snaps first to find the max/min)
v_global_min = np.inf
v_global_max = -np.inf
M_arrays = [] 

for i in range(4):
    U = v_flds_sum[i][0][::step, ::step, slice_z]
    V = v_flds_sum[i][1][::step, ::step, slice_z]
    M = np.hypot(U, V)
    
    log_M = np.log10(M)
    M_arrays.append((U, V, M, log_M, M)) # Store them so we don't recalculate
    
    v_global_max = max(v_global_max, np.nanmax(log_M))
    v_global_min = min(v_global_min, np.nanmin(log_M))

# --- STEP 2: PLOT THE GRIDS ---
for i in range(4):
    ax = axes[i]
    
    # Plot Density
    total = nodes[cute_id[i]]*4 + filaments[cute_id[i]]*3 + walls[cute_id[i]]*2 + voids[cute_id[i]]
    im = ax.imshow(total[:,:,slice_z], origin="lower", cmap="viridis_r", vmax = 4, vmin = 1, extent=[0, L, 0, L])

    # Inside your plotting loop (Step 2)
    U, V, M, log_M, M = M_arrays[i]
    
    # X and Y must be 2D grids for streamplot, matching U and V
    X_grid, Y_grid = np.meshgrid(np.linspace(0, 35, 128)[::step], 
                                 np.linspace(0, 35, 128)[::step])
    
    # Plot Streamlines instead of Quiver
    strm = ax.streamplot(X_grid, Y_grid, U, V, 
                         color="black",           # Color mapped to log magnitude
                         #cmap='inferno', 
                         norm=plt.Normalize(vmin=v_global_min, vmax=v_global_max),
                         linewidth = M/200,         # Can also map linewidth to magnitude!
                         density=1.75)           # Controls how packed the lines are

    ax.set_aspect("equal")
    #ax.set_title(fr"$z={z_list_cute[i]}$", fontsize = 17)

    ax.text(0.04, 0.96,                  # X and Y position
    f"$z = {z_list_cute[i]}$",   # The text to display
    transform=ax.transAxes,      # Use axis coordinates (0 to 1) instead of data coordinates
    fontsize=20, 
    color="white", 
    verticalalignment="top", 
    bbox=dict(facecolor="black", alpha=0.75, edgecolor="none", pad=5)
    )

    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()

plt.savefig("nexus_streams.png", transparent=True, bbox_inches="tight")
plt.savefig("nexus_streams.pdf", transparent=True, bbox_inches="tight");


In [ ]:
aaa = 1/(np.array(red) +1)

In [ ]:
print(aaa)

In [ ]:
fig = plt.figure(figsize=(11,4))
ax = fig.add_subplot(121)


styles = ["solid","-.","--",":"]

cos_bins = np.linspace(-1, 1, 100) 
heatmap_data = np.zeros((len(cos_bins)-1, 100))

for t in range(100):

    hist, _ = np.histogram(cos_all[0][t], bins=cos_bins)
    heatmap_data[:, t] = hist

ax.plot(aaa, cos_avg[0], label = "Average", c="black", linewidth = 2)

aaa_centers = (aaa[:-1] + aaa[1:]) / 2
cos_centers = (cos_bins[:-1] + cos_bins[1:]) / 2

pcm = ax.pcolormesh(
    aaa_centers, 
    cos_centers, 
    heatmap_data[:,:-1] + 1,  # Trims off the extra 100th column to match the 99 centers
    shading='auto', 
    cmap='magma', norm = LogNorm()
)
ax.legend()

ax.set_ylabel(r"$\cos \theta$")
ax.set_xlabel(r"Scale factor $a$")

ax.set_xlim(aaa.min(), aaa.max())
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(f"Particle Count")

def a2z(a):
    return 1/a - 1


def z2a(z):
    return 1/(1+z)


secax = ax.secondary_xaxis('top', functions=(a2z, z2a))
secax.set_xlabel(r'Redshift $z$')
secax.set_xticks([10, 5, 3, 2, 1, 0.5, 0])




ax = fig.add_subplot(122)

for k in range(1,5):
    ax.plot(aaa, cos_avg[k], label = f"Avg for {struct[k]}", linestyle = styles[k-1], c = colors[k-1])

        
ax.set_xlabel(r"Scale factor $a$")
ax.set_ylabel(r"$\cos \theta$")
ax.legend()
ax.grid()
secax = ax.secondary_xaxis('top', functions=(a2z, z2a))
secax.set_xlabel(r'Redshift $z$')
secax.set_xticks([10, 5, 3, 2, 1, 0.5, 0])

plt.tight_layout()

plt.savefig("cos.png", transparent = True, bbox_inches="tight")
plt.savefig("cos.pdf", transparent = True, bbox_inches="tight");

In [ ]:
fig = plt.figure(figsize=(11,4))
ax = fig.add_subplot(121)


styles = ["solid","-.","--",":"]

mag_bins = np.linspace(np.min(np.log10(mag_all[0])), np.max(np.log10(mag_all[0])), 100) 
heatmap_data = np.zeros((len(mag_bins)-1, 100))

for t in range(100):

    hist, _ = np.histogram(np.log10(mag_all[0][t]), bins=mag_bins)
    heatmap_data[:, t] = hist

ax.plot(aaa, np.log10(mag_avg[0]), label = "Average", c="black", linewidth = 2)

aaa_centers = (aaa[:-1] + aaa[1:]) / 2
mag_centers = (mag_bins[:-1] + mag_bins[1:]) / 2

pcm = ax.pcolormesh(
    aaa_centers, 
    mag_centers, 
    heatmap_data[:,:-1] + 1,  # Trims off the extra 100th column to match the 99 centers
    shading='auto', 
    cmap='plasma', norm = LogNorm()
)
ax.legend()

ax.set_ylabel(r"$\log(|v_x|/|v_q|)$")
ax.set_xlabel(r"Scale factor $a$")

ax.set_xlim(aaa.min(), aaa.max())
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(f"Particle Count")

def a2z(a):
    return 1/a - 1


def z2a(z):
    return 1/(1+z)


secax = ax.secondary_xaxis('top', functions=(a2z, z2a))
secax.set_xlabel(r'Redshift $z$')
secax.set_xticks([10, 5, 3, 2, 1, 0.5, 0])




ax = fig.add_subplot(122)

for k in range(1,5):
    ax.plot(aaa, mag_avg[k], label = f"Avg for {struct[k]}", linestyle = styles[k-1], c = colors[k-1])

        
ax.set_xlabel(r"Scale factor $a$")
ax.set_ylabel(r"$|v_x|/|v_q|$")
ax.legend()
ax.grid()
secax = ax.secondary_xaxis('top', functions=(a2z, z2a))
secax.set_xlabel(r'Redshift $z$')
secax.set_xticks([10, 5, 3, 2, 1, 0.5, 0])


plt.tight_layout()

plt.savefig("mag.png", transparent = True, bbox_inches="tight")
plt.savefig("mag.pdf", transparent = True, bbox_inches="tight");

In [ ]:
fig = plt.figure(figsize=(11,4))
ax = fig.add_subplot(121)


styles = ["solid","-.","--",":"]

mag_bins = np.linspace(np.min(mag_all[0]), np.max(mag_all[0]), 100) 
heatmap_data = np.zeros((len(mag_bins)-1, 100))

for t in range(100):

    hist, _ = np.histogram(mag_all[0][t], bins=mag_bins)
    heatmap_data[:, t] = hist

ax.plot(aaa, mag_avg[0], label = "Average", c="black", linewidth = 2)

aaa_centers = (aaa[:-1] + aaa[1:]) / 2
mag_centers = (mag_bins[:-1] + mag_bins[1:]) / 2

pcm = ax.pcolormesh(
    aaa_centers, 
    mag_centers, 
    heatmap_data[:,:-1] + 1,  # Trims off the extra 100th column to match the 99 centers
    shading='auto', 
    cmap='magma', norm = LogNorm()
)
ax.legend()

ax.set_ylabel(r"$|v_x|/|v_q|$")
ax.set_xlabel(r"Scale factor $a$")
ax.set_ylim(0,8)

ax.set_xlim(aaa.min(), aaa.max())
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(f"Particle Count")

def a2z(a):
    return 1/a - 1


def z2a(z):
    return 1/(1+z)


secax = ax.secondary_xaxis('top', functions=(a2z, z2a))
secax.set_xlabel(r'Redshift $z$')
secax.set_xticks([10, 5, 3, 2, 1, 0.5, 0])




ax = fig.add_subplot(122)

for k in range(1,5):
    ax.plot(aaa, mag_avg[k], label = f"Avg for {struct[k]}", linestyle = styles[k-1], c = colors[k-1])

        
ax.set_xlabel(r"Scale factor $a$")
ax.set_ylabel(r"$|v_x|/|v_q|$")
ax.legend()
ax.grid()
secax = ax.secondary_xaxis('top', functions=(a2z, z2a))
secax.set_xlabel(r'Redshift $z$')
secax.set_xticks([10, 5, 3, 2, 1, 0.5, 0])


plt.tight_layout()

plt.savefig("mag.png", transparent = True, bbox_inches="tight")
plt.savefig("mag.pdf", transparent = True, bbox_inches="tight");

In [ ]:
def huhuhehe(snap_curr):
    # 1. Setup paths, snapshots, and parameters
    basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"
    snap_init = 0   # Initial snapshot (z=20.05)
    #snap_curr = 99  # Current snapshot (z=0.0)
    part_type = 1   # 1 is for Dark Matter
    fraction = 0.01 # Process only 10% of the particles

    print(f"Working on {snap_curr}")

    # 2. Load Data (Only loading the specific fields we need to save memory)
    fields_init = ['ParticleIDs', 'Velocities']
    data_init = il.snapshot.loadSubset(basePath, snap_init, part_type, fields=fields_init)
    id_init =  data_init['ParticleIDs']
    v_init = data_init['Velocities']

    fields_curr = ['ParticleIDs', 'Velocities', 'Coordinates']
    data_curr = il.snapshot.loadSubset(basePath, snap_curr, part_type, fields=fields_curr)
    id_curr = data_curr['ParticleIDs']
    v_curr = data_curr['Velocities']
    pos_curr = data_curr['Coordinates']/1000


    # 3. Sort both arrays so the same particle is at the exact same index
    sort_init = np.argsort(id_init)
    sort_curr = np.argsort(id_curr)

    v_init_sorted = v_init[sort_init]
    v_curr_sorted = v_curr[sort_curr]
    pos_curr_sorted = pos_curr[sort_curr]

    # Free up memory from the unsorted arrays
    del data_init, data_curr, id_init, id_curr, v_init, v_curr, pos_curr, sort_init, sort_curr


    # 4. Subsample the particles
    num_particles = len(v_curr_sorted)
    num_sample = int(num_particles * fraction)

    # Randomly select indices without replacement
    np.random.seed(42) # Set seed so your plots are reproducible
    sample_indices = np.random.choice(num_particles, num_sample, replace=False)

    v_init_sampled = v_init_sorted[sample_indices]
    v_curr_sampled = v_curr_sorted[sample_indices]
    pos_curr_sampled = pos_curr_sorted[sample_indices]


    # 5. Compute the Dot Product and Magnitudes
    # np.sum with axis=1 multiplies element-wise and sums the x, y, z components
    dot_product = np.sum(v_init_sampled * v_curr_sampled, axis=1)

    # Calculate magnitudes (adding a tiny number 1e-10 to prevent division by zero errors)
    norm_init = np.linalg.norm(v_init_sampled, axis=1) + 1e-10
    norm_curr = np.linalg.norm(v_curr_sampled, axis=1) + 1e-10

    # Calculate cosine
    cos_theta = dot_product / (norm_init * norm_curr)


    # 6. Define a slice
    # Taking a slice along the Z-axis between 0.0 and 10.0 cMpc/h (converted to ckpc/h)
    z_min = 0.0 
    z_max = 12.5 

    # Extract Z coordinates and create a boolean mask
    z_coords = pos_curr_sampled[:, 2]
    slice_mask = (z_coords >= z_min) & (z_coords <= z_max)

    # Apply the mask
    x_slice = pos_curr_sampled[:, 0][slice_mask]
    y_slice = pos_curr_sampled[:, 1][slice_mask]
    cos_slice = cos_theta[slice_mask]

    return x_slice, y_slice, cos_slice

In [ ]:
def magnitude(snap_curr):
    # 1. Setup paths, snapshots, and parameters
    basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"
    snap_init = 0   # Initial snapshot (z=20.05)
    #snap_curr = 99  # Current snapshot (z=0.0)
    part_type = 1   # 1 is for Dark Matter
    fraction = 0.01 # Process only 10% of the particles

    print(f"Working on {snap_curr}")

    # 2. Load Data (Only loading the specific fields we need to save memory)
    fields_init = ['ParticleIDs', 'Velocities']
    data_init = il.snapshot.loadSubset(basePath, snap_init, part_type, fields=fields_init)
    id_init =  data_init['ParticleIDs']
    v_init = data_init['Velocities']

    fields_curr = ['ParticleIDs', 'Velocities', 'Coordinates']
    data_curr = il.snapshot.loadSubset(basePath, snap_curr, part_type, fields=fields_curr)
    id_curr = data_curr['ParticleIDs']
    v_curr = data_curr['Velocities']
    pos_curr = data_curr['Coordinates']/1000


    # 3. Sort both arrays so the same particle is at the exact same index
    sort_init = np.argsort(id_init)
    sort_curr = np.argsort(id_curr)

    v_init_sorted = v_init[sort_init]
    v_curr_sorted = v_curr[sort_curr]
    pos_curr_sorted = pos_curr[sort_curr]

    # Free up memory from the unsorted arrays
    del data_init, data_curr, id_init, id_curr, v_init, v_curr, pos_curr, sort_init, sort_curr


    # 4. Subsample the particles
    num_particles = len(v_curr_sorted)
    num_sample = int(num_particles * fraction)

    # Randomly select indices without replacement
    np.random.seed(42) # Set seed so your plots are reproducible
    sample_indices = np.random.choice(num_particles, num_sample, replace=False)

    v_init_sampled = v_init_sorted[sample_indices]
    mag_0 = np.sqrt(v_init_sampled[:,0]**2 + v_init_sampled[:,1]**2 + v_init_sampled[:,2]**2) + 1e-10
    v_curr_sampled = v_curr_sorted[sample_indices] 
    pos_curr_sampled = pos_curr_sorted[sample_indices]
    mag = np.sqrt(v_curr_sampled[:,0]**2 + v_curr_sampled[:,1]**2 + v_curr_sampled[:,2]**2) + 1e-10


    # 5. Compute the Dot Product and Magnitudes
    # np.sum with axis=1 multiplies element-wise and sums the x, y, z components
    factor = mag/mag_0

    # 6. Define a slice
    # Taking a slice along the Z-axis between 0.0 and 10.0 cMpc/h (converted to ckpc/h)
    z_min = 0.0 
    z_max = 12.5 

    # Extract Z coordinates and create a boolean mask
    z_coords = pos_curr_sampled[:, 2]
    slice_mask = (z_coords >= z_min) & (z_coords <= z_max)

    # Apply the mask
    x_slice = pos_curr_sampled[:, 0][slice_mask]
    y_slice = pos_curr_sampled[:, 1][slice_mask]
    factor_slice = factor[slice_mask]

    return x_slice, y_slice, factor_slice

In [ ]:
data = []

for i in [99, 33, 17, 4]:
    data.append(huhuhehe(i))

In [ ]:
data2 = []

for i in [99, 33, 17, 4]:
    data2.append(magnitude(i))

In [ ]:
fig = plt.figure(figsize = (10,8))

for i in range(4):
    ax = fig.add_subplot(2,2,i+1)
    im = ax.scatter(
        data[i][0], 
        data[i][1], 
        c=data[i][2], 
        cmap='cividis', 
        vmin=-1, 
        vmax=1, 
        s=0.1)
    
    fig.colorbar(im, ax = ax, label = r"$\cos \theta$")
    ax.set_title(fr"$z = {z_list_cute[i]}$")
    ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.set_xlabel("x [cMpc/h]")
    ax.set_ylabel("y [cMpc/h]")
    ax.set_xlim(0, 35)
    ax.set_ylim(0, 35)

plt.tight_layout()

plt.savefig("cos_map.png", transparent=True, bbox_inches="tight")
plt.savefig("cos_map.pdf", transparent=True, bbox_inches="tight");

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. ADD layout="constrained" HERE. This perfectly handles global colorbars.
fig, axes = plt.subplots(2, 2, figsize=(14, 12), layout="constrained")
axes = axes.flatten() 

cute_id = [0, 9, 12, 17]
z_list_cute = [0.0, 2.0, 5.0, 10.0] 
slice_z = 13
step = 5
L = 35 # Assuming L is defined somewhere in your code

# --- STEP 1: CALCULATE GLOBAL LIMITS ---
# (Assuming norm_dens is defined elsewhere)
# global_min = np.min([np.log10(norm_dens[cute_id[i]][:,:,slice_z]) for i in range(4)])
# global_max = np.max([np.log10(norm_dens[cute_id[i]][:,:,slice_z]) for i in range(4)])

# --- STEP 2: PLOT THE GRIDS ---
for i in range(4):
    ax = axes[i] # <--- FIX 1: Use the axis that plt.subplots already created!

    im = ax.scatter(
        data[i][0], 
        data[i][1], 
        c=data[i][2], 
        cmap='cividis', 
        vmin=-1, 
        vmax=1, 
        s=0.1
    )
    
    ax.set_aspect("equal")

    # # <--- FIX 2: THIS REMOVES THE EVIL BORDERS --->
    # for spine in ax.spines.values():
    #     spine.set_visible(False)
    # # <-------------------------------------------->

    ax.text(0.04, 0.96,                  
            f"$z = {z_list_cute[i]}$",   
            transform=ax.transAxes,      
            fontsize=20, 
            color="white", 
            verticalalignment="top", 
            bbox=dict(facecolor="black", alpha=0.75, edgecolor="none", pad=5))

    # Keep these to remove the tick marks as well
    ax.set_xticks([])
    ax.set_yticks([])

# --- STEP 3: THE RIGHT WAY TO ADD THE GLOBAL COLORBAR ---
cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.9, aspect=30)
cbar.set_label(r"$\cos \theta$", fontsize=20)
cbar.ax.tick_params(labelsize=15) 

plt.savefig("cos_map.png", transparent=True, bbox_inches="tight", dpi=300)
plt.savefig("cos_map.pdf", transparent=True, bbox_inches="tight")

In [ ]:
import matplotlib.colors as mcolors

fig = plt.figure(figsize = (10,8))

v_min = np.percentile([np.nanmin(np.log10(data2[i][2])) for i in range(4)],3)
v_max = np.percentile([np.nanmax(np.log10(data2[i][2])) for i in range(4)],97)

#custom_norm = mcolors.TwoSlopeNorm(vmin=v_min, vcenter=0, vmax=v_max)
custom_norm = mcolors.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

for i in range(4):
    ax = fig.add_subplot(2,2,i+1)
    im = ax.scatter(
        data2[i][0], 
        data2[i][1], 
        c=np.log10(data2[i][2]), 
        cmap='seismic_r', 
        s=0.1, norm = custom_norm)
    
    fig.colorbar(im, ax = ax, label = r"$\log (|v_x|/|v_q|)$")
    ax.set_title(fr"$z = {z_list_cute[i]}$")
    ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.set_xlabel("x [cMpc/h]")
    ax.set_ylabel("y [cMpc/h]")
    ax.set_xlim(0, 35)
    ax.set_ylim(0, 35)

plt.tight_layout()

plt.savefig("mag_map.png", transparent=True, bbox_inches="tight")
plt.savefig("mag_map.pdf", transparent=True, bbox_inches="tight");

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. ADD layout="constrained" HERE. This perfectly handles global colorbars.
fig, axes = plt.subplots(2, 2, figsize=(14, 12), layout="constrained")
axes = axes.flatten() 

import matplotlib.colors as mcolors

v_min = np.percentile([np.nanmin(np.log10(data2[i][2])) for i in range(4)],3)
v_max = np.percentile([np.nanmax(np.log10(data2[i][2])) for i in range(4)],97)

#custom_norm = mcolors.TwoSlopeNorm(vmin=v_min, vcenter=0, vmax=v_max)
custom_norm = mcolors.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

cute_id = [0, 9, 12, 17]
z_list_cute = [0.0, 2.0, 5.0, 10.0] 
slice_z = 13
step = 5
L = 35 # Assuming L is defined somewhere in your code

# --- STEP 1: CALCULATE GLOBAL LIMITS ---
# (Assuming norm_dens is defined elsewhere)
# global_min = np.min([np.log10(norm_dens[cute_id[i]][:,:,slice_z]) for i in range(4)])
# global_max = np.max([np.log10(norm_dens[cute_id[i]][:,:,slice_z]) for i in range(4)])

# --- STEP 2: PLOT THE GRIDS ---
for i in range(4):
    ax = axes[i] # <--- FIX 1: Use the axis that plt.subplots already created!

    im = ax.scatter(
        data2[i][0], 
        data2[i][1], 
        c=np.log10(data2[i][2]), 
        cmap='seismic_r',  
        s=0.1, norm = custom_norm
    )
    
    ax.set_aspect("equal")

    # # <--- FIX 2: THIS REMOVES THE EVIL BORDERS --->
    # for spine in ax.spines.values():
    #     spine.set_visible(False)
    # # <-------------------------------------------->

    ax.text(0.04, 0.96,                  
            f"$z = {z_list_cute[i]}$",   
            transform=ax.transAxes,      
            fontsize=20, 
            color="white", 
            verticalalignment="top", 
            bbox=dict(facecolor="black", alpha=0.75, edgecolor="none", pad=5))

    # Keep these to remove the tick marks as well
    ax.set_xticks([])
    ax.set_yticks([])

# --- STEP 3: THE RIGHT WAY TO ADD THE GLOBAL COLORBAR ---
cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.9, aspect=30)
cbar.set_label(r"$\log (|v_x|/|v_q|)$", fontsize=20)
cbar.ax.tick_params(labelsize=15) 

plt.savefig("mag_map.png", transparent=True, bbox_inches="tight", dpi=300)
plt.savefig("mag_map.pdf", transparent=True, bbox_inches="tight")